# Exercise XP: Air Traffic Data Analysis

**Course:** Developers Institute  **Week 5 - Day 3**  
**Author:** Alex Goldbaum

Inferential statistics + regression on air traffic data. Eight sections:
Setup, EDA, Hypothesis Testing, Simple Linear Regression, Multiple Linear
Regression, Model Comparison, Insights, and Reflection. The notebook loads
`air_traffic_data.csv` if present, otherwise generates a realistic sample.


## Section 1 — Setup and Data Loading


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [ ]:
# Load the dataset. If the file is not found, generate a realistic sample so the
# notebook is self-contained (this is the fallback the template suggests).
import os

CSV_PATH = 'air_traffic_data.csv'

if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
    print(f'Loaded real data from {CSV_PATH}')
else:
    print('CSV not found — generating sample air-traffic data (200 months).')
    n = 200
    # Simulate a market that grows over time with seasonality + noise
    t = np.arange(n)
    season = 1 + 0.10 * np.sin(2 * np.pi * t / 12)
    trend = 1 + 0.003 * t

    dom_pax = np.random.normal(60_000_000, 4_000_000, n) * season * trend
    int_pax = np.random.normal(20_000_000, 2_500_000, n) * season * trend
    pax = dom_pax + int_pax

    # Flights are tightly coupled with passengers (~load factor + noise)
    dom_flt = dom_pax / np.random.normal(110, 5, n) + np.random.normal(0, 3000, n)
    int_flt = int_pax / np.random.normal(220, 10, n) + np.random.normal(0, 1500, n)
    flt = dom_flt + int_flt

    # Revenue Passenger-Miles (Domestic) — Dom_Pax × average trip length
    dom_rpm = dom_pax * np.random.normal(950, 60, n)

    df = pd.DataFrame({
        'Dom_Pax': dom_pax.round().astype(int),
        'Int_Pax': int_pax.round().astype(int),
        'Pax':     pax.round().astype(int),
        'Dom_Flt': dom_flt.round().astype(int),
        'Int_Flt': int_flt.round().astype(int),
        'Flt':     flt.round().astype(int),
        'Dom_RPM': dom_rpm.round().astype(int),
    })

print(f'Shape: {df.shape}')
df.head()


## Section 2 — Exploratory Data Analysis


In [ ]:
# Dataset info
df.info()


In [ ]:
# Statistical summary
df.describe().round(0)


In [ ]:
# Missing values
missing = df.isnull().sum()
print('Missing values per column:')
print(missing)
print(f'\nTotal missing cells: {missing.sum()}')


In [ ]:
# Correlation matrix + heatmap
corr = df.corr()

plt.figure(figsize=(9, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5)
plt.title('Correlation matrix — Air traffic variables', fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Identify the strongest correlations (|r| > 0.7), excluding the diagonal.
pairs = []
for a in corr.columns:
    for b in corr.columns:
        if a < b:  # avoid duplicates + self-correlation
            pairs.append((a, b, corr.loc[a, b]))
strong = pd.DataFrame(pairs, columns=['Var A', 'Var B', 'Correlation'])
strong['|r|'] = strong['Correlation'].abs()
strong = strong.sort_values('|r|', ascending=False).drop(columns='|r|').reset_index(drop=True)
print('Strongest correlations:')
print(strong.round(3))


**EDA findings.**
- `Pax`, `Dom_Pax`, `Int_Pax`, `Flt`, `Dom_Flt`, `Int_Flt`, `Dom_RPM` are all
  strongly correlated — flights and passengers move together, which is the
  expected operational signal.
- `Pax = Dom_Pax + Int_Pax` and `Flt = Dom_Flt + Int_Flt` by construction —
  these are **definitional identities** and create perfect multicollinearity if
  we feed both into a regression. We will exclude `Pax` and `Flt` as predictors
  for the multiple-regression model.
- `Dom_RPM` correlates highly with `Dom_Pax` because RPM ≈ passengers × distance,
  but it also carries trip-length information — useful as a feature.


## Section 3 — Hypothesis Testing

Significance level: **α = 0.05**.


### Test 1 — Domestic vs International passengers (independent t-test)

- **H₀:** mean Dom_Pax = mean Int_Pax
- **H₁:** mean Dom_Pax ≠ mean Int_Pax


In [ ]:
t_stat, p_value = stats.ttest_ind(df['Dom_Pax'], df['Int_Pax'])

print(f't-statistic: {t_stat:.4f}')
print(f'p-value:     {p_value:.6g}')
print(f'Mean Dom_Pax: {df["Dom_Pax"].mean():,.0f}')
print(f'Mean Int_Pax: {df["Int_Pax"].mean():,.0f}')

alpha = 0.05
if p_value < alpha:
    print(f'\nDecision: p < {alpha} -> REJECT H0. The means are significantly different.')
else:
    print(f'\nDecision: p >= {alpha} -> fail to reject H0.')


**Interpretation.** Domestic and international passenger volumes come from very
different distributions: domestic traffic is roughly 3× larger than international
in this market. The t-test confirms the difference is **statistically significant**
(p ≪ 0.05) — and the **practical** gap (~40 M passengers per month) is also large,
so the difference matters for operations and capacity planning.


### Test 2 — Correlation between total passengers and total flights (Pearson)

- **H₀:** ρ = 0 (no linear correlation)
- **H₁:** ρ ≠ 0


In [ ]:
r, p_value_corr = stats.pearsonr(df['Pax'], df['Flt'])

print(f'Pearson r: {r:.4f}')
print(f'p-value:   {p_value_corr:.6g}')

if p_value_corr < alpha:
    direction = 'positive' if r > 0 else 'negative'
    strength = 'strong' if abs(r) > 0.7 else 'moderate' if abs(r) > 0.4 else 'weak'
    print(f'\nDecision: REJECT H0. There is a {strength} {direction} correlation '
          f'(r = {r:.3f}).')
else:
    print('\nDecision: fail to reject H0 - no significant correlation.')


**Interpretation.** The relationship between `Pax` and `Flt` is strong and
positive (r ≈ 0.99). Operationally this is unsurprising — more flights move more
passengers — and it tells us that a linear regression on flights alone should
already predict total passengers very well.


## Section 4 — Simple Linear Regression

Model: predict `Pax` from `Flt`.


In [ ]:
X = df[['Flt']]
y = df['Pax']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

simple_model = LinearRegression()
simple_model.fit(X_train, y_train)

intercept = simple_model.intercept_
coef = simple_model.coef_[0]
print(f'Model equation: Pax = {intercept:,.0f} + {coef:.2f} * Flt')


In [ ]:
y_pred_simple = simple_model.predict(X_test)

r2_simple = r2_score(y_test, y_pred_simple)
mse_simple = mean_squared_error(y_test, y_pred_simple)
rmse_simple = np.sqrt(mse_simple)
mae_simple = mean_absolute_error(y_test, y_pred_simple)

print(f'R² Score: {r2_simple:.4f}')
print(f'MSE     : {mse_simple:,.0f}')
print(f'RMSE    : {rmse_simple:,.0f}')
print(f'MAE     : {mae_simple:,.0f}')


In [ ]:
# Visualize: scatter + regression line; residual plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(X_test, y_test, alpha=0.6, color='steelblue', label='Actual')
x_range = np.linspace(X_test.min().iloc[0], X_test.max().iloc[0], 100).reshape(-1, 1)
axes[0].plot(x_range, simple_model.predict(x_range), 'r-', linewidth=2, label='Regression line')
axes[0].set_xlabel('Total Flights (Flt)')
axes[0].set_ylabel('Total Passengers (Pax)')
axes[0].set_title(f'Simple Linear Regression — R² = {r2_simple:.3f}', fontweight='bold')
axes[0].legend()

residuals = y_test - y_pred_simple
axes[1].scatter(y_pred_simple, residuals, alpha=0.6, color='seagreen')
axes[1].axhline(0, color='black', linestyle='--', linewidth=1)
axes[1].set_xlabel('Predicted Pax')
axes[1].set_ylabel('Residuals (Actual - Predicted)')
axes[1].set_title('Residual plot', fontweight='bold')

plt.tight_layout()
plt.show()


## Section 5 — Multiple Linear Regression

Features: `Dom_Pax`, `Int_Pax`, `Dom_Flt`, `Int_Flt`, `Dom_RPM`. We exclude `Pax`
(it IS the target) and `Flt` (it is the sum of `Dom_Flt + Int_Flt`, so feeding it
in creates perfect multicollinearity).


In [ ]:
feature_cols = ['Dom_Pax', 'Int_Pax', 'Dom_Flt', 'Int_Flt', 'Dom_RPM']
X_multi = df[feature_cols]
y_multi = df['Pax']

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_multi, y_multi, test_size=0.2, random_state=RANDOM_STATE
)

# Standardize: fit only on train (no leakage), transform both
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_m)
X_test_scaled = scaler.transform(X_test_m)

multi_model = LinearRegression()
multi_model.fit(X_train_scaled, y_train_m)

coef_df = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient (scaled)': multi_model.coef_,
}).sort_values('Coefficient (scaled)', key=abs, ascending=False)
print('Coefficients (on standardized features — larger |value| = more influential):')
print(coef_df.round(0))
print(f'\nIntercept: {multi_model.intercept_:,.0f}')


In [ ]:
y_pred_multi = multi_model.predict(X_test_scaled)

r2_multi = r2_score(y_test_m, y_pred_multi)
mse_multi = mean_squared_error(y_test_m, y_pred_multi)
rmse_multi = np.sqrt(mse_multi)
mae_multi = mean_absolute_error(y_test_m, y_pred_multi)

print(f'R² Score: {r2_multi:.6f}')
print(f'MSE     : {mse_multi:,.0f}')
print(f'RMSE    : {rmse_multi:,.0f}')
print(f'MAE     : {mae_multi:,.0f}')


## Section 6 — Model Comparison and Analysis


In [ ]:
comparison = pd.DataFrame({
    'Metric': ['R²', 'MSE', 'RMSE', 'MAE'],
    'Simple Regression': [r2_simple, mse_simple, rmse_simple, mae_simple],
    'Multiple Regression': [r2_multi, mse_multi, rmse_multi, mae_multi],
})

# Improvement % — for R² higher is better, for the errors lower is better
def improvement_pct(old, new, higher_is_better):
    if higher_is_better:
        return (new - old) / abs(old) * 100
    else:
        return (old - new) / abs(old) * 100

improvements = [
    improvement_pct(r2_simple, r2_multi, higher_is_better=True),
    improvement_pct(mse_simple, mse_multi, higher_is_better=False),
    improvement_pct(rmse_simple, rmse_multi, higher_is_better=False),
    improvement_pct(mae_simple, mae_multi, higher_is_better=False),
]
comparison['Improvement %'] = improvements

print(comparison.round(4))

winner = 'Multiple Regression' if r2_multi > r2_simple else 'Simple Regression'
print(f'\nBetter model (by R²): {winner}')


In [ ]:
# Visual comparison
fig, ax = plt.subplots(figsize=(9, 5))
metrics = ['R²']  # plot R² alone (the errors have very different magnitudes)
x = np.arange(len(metrics))
width = 0.35
ax.bar(x - width/2, [r2_simple], width, label='Simple', color='steelblue')
ax.bar(x + width/2, [r2_multi], width, label='Multiple', color='seagreen')
ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.set_ylim(0.9, 1.0001)
ax.set_title('R² — Simple vs Multiple Linear Regression', fontweight='bold')
ax.legend()
for i, v in enumerate([r2_simple, r2_multi]):
    ax.text((i - 0.5) * width, v + 0.001, f'{v:.4f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()


## Section 7 — Statistical Insights and Conclusions

**Hypothesis testing.**
- Domestic and international passenger volumes are statistically (and practically)
  different. Operationally this means domestic and international should be planned
  as **separate capacity pools** — peaks, seasonality and recovery dynamics are
  not the same.
- Total passengers and total flights show a near-perfect positive correlation
  (r ≈ 0.99). Flights are an excellent proxy for traffic load.

**Regression results.**
- The simple regression on `Flt` alone already explains a very high share of the
  variance in `Pax` (R² ≈ 0.98+). This is the practical takeaway: in this market,
  one variable does most of the explanatory work.
- The multiple regression with `Dom_Pax`, `Int_Pax`, `Dom_Flt`, `Int_Flt`,
  `Dom_RPM` reaches R² ≈ 1.000 because two of its features (`Dom_Pax + Int_Pax`)
  literally sum to the target — the model is recovering an algebraic identity.
- This means the multiple model is *technically* better on every metric, but the
  improvement comes from definitional leakage, not from real predictive power.

**Correlations.**
- The strongest non-trivial signal is between flights and passengers — load factors
  are stable. A jump in flights without a matching jump in passengers would signal
  capacity overshoot.
- `Dom_RPM` carries information about average trip length on top of passenger
  counts, useful for revenue planning.

**Recommendations.**
1. Use flights as a leading operational KPI for forecasting passenger demand.
2. Plan domestic and international capacity separately — different scales,
   different seasonality.
3. Watch the `Pax / Flt` ratio (load factor) as an early warning of efficiency
   loss when supply outpaces demand.
4. When building production-grade forecasts, **never** include features that are
   algebraic sums of the target (avoid `Pax` being predicted from `Dom_Pax +
   Int_Pax`). Predict from genuinely exogenous variables: prior-month traffic,
   GDP, fuel prices, fare indices, holiday calendars.


## Section 8 — Reflection Questions

**1. What do hypothesis test results reveal about air traffic patterns?**
The t-test shows domestic and international are two distinct populations with very
different scales. The Pearson test shows the passenger-flight relationship is so
tight that flights can be used as a near-perfect operational proxy for traffic.
Both results match what we expect from real airline data.

**2. Why did one regression model perform better than the other?**
On paper, the multiple regression reaches R² ≈ 1.000 — but the reason is *target
leakage*: the predictors `Dom_Pax + Int_Pax` literally sum to the target `Pax`,
so the model is recovering an identity, not learning a relationship. The simple
regression on `Flt` is the more honest model — it generalizes from a real
physical signal (more flights ⇒ more passengers).

**3. How can airlines use correlation insights operationally?**
- Flight counts as a leading indicator for passenger forecasting.
- Domestic and international planned as separate capacity pools.
- Monitoring load factor (Pax / Flt) for efficiency changes.
- RPM as input for revenue forecasting (it embeds trip-length information).

**4. What do residual plots tell you about model assumptions?**
If residuals are randomly scattered around zero with constant variance, the linear
regression assumptions are satisfied. Curved or fanning patterns would suggest
non-linearity or heteroscedasticity — in either case a transformation
(log of passengers, polynomial features) or a non-linear model would help.

**5. What are practical applications of these statistical models?**
Demand forecasting, capacity planning, route profitability analysis, dynamic
pricing, fuel hedging, ground-staff scheduling, airport slot allocation. The
same workflow (EDA → hypothesis test → simple model → multiple model → critique)
applies to any operational forecasting problem in transportation, retail or
logistics.
